In [2]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# --- 1. Simulated Financial Data ---
# Suppose we have 4 assets with expected annual returns and a covariance matrix
num_assets = 4
expected_returns = np.array([0.12, 0.15, 0.08, 0.10])
cov_matrix = np.array([
    [0.05, 0.02, 0.01, 0.01],
    [0.02, 0.08, 0.02, 0.01],
    [0.01, 0.02, 0.04, 0.005],
    [0.01, 0.01, 0.005, 0.03],
])
risk_free_rate = 0.03


# --- 2. Fitness Function (Sharpe Ratio) ---
def evaluate_fitness(weights):
  # Ensure weights sum to 1
  weights = weights / np.sum(weights)
  port_return = np.dot(weights, expected_returns)
  port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
  sharpe_ratio = (port_return - risk_free_rate) / port_volatility
  return sharpe_ratio


# --- 3. Genetic Algorithm Parameters ---
pop_size = 50
num_generations = 100
mutation_rate = 0.1

# Initialize random population (weights)
population = np.random.rand(pop_size, num_assets)
population = population / np.sum(population, axis=1, keepdims=True)

# --- 4. Evolution Loop ---
for generation in range(num_generations):
  # Evaluate fitness for each individual in the population
  fitness_scores = np.array([evaluate_fitness(ind) for ind in population])

  # Select top 50% parents using tournament or rank selection
  sorted_indices = np.argsort(fitness_scores)[::-1]
  population = population[sorted_indices]
  parents = population[: pop_size // 2]

  # Crossover: combine pairs of parents to create offspring
  offspring = []
  while len(offspring) < (pop_size - len(parents)):
    p1, p2 = parents[np.random.choice(len(parents), 2, replace=False)]
    crossover_point = np.random.randint(1, num_assets - 1)
    child = np.concatenate([p1[:crossover_point], p2[crossover_point:]])
    offspring.append(child)

  offspring = np.array(offspring)

  # Mutation: apply random noise to offspring
  mutation_mask = np.random.rand(*offspring.shape) < mutation_rate
  offspring += mutation_mask * np.random.normal(0, 0.1, offspring.shape)
  offspring = np.abs(offspring)  # Keep weights positive
  offspring = offspring / np.sum(offspring, axis=1, keepdims=True)

  # Form new population
  population = np.vstack([parents, offspring])

# --- 5. Best Solution ---
final_fitness = np.array([evaluate_fitness(ind) for ind in population])
best_idx = np.argmax(final_fitness)
best_weights = population[best_idx] / np.sum(population[best_idx])

print("Optimal Asset Weights:", np.round(best_weights, 4))
print("Max Sharpe Ratio:", round(final_fitness[best_idx], 4))


Optimal Asset Weights: [0.2626 0.2458 0.0789 0.4127]
Max Sharpe Ratio: 0.5804
